# step5 — 지침 신호 인과 (RQ3 확정)

**무엇을 확인하나:** 지침의 지시어 단어(예: `camelCase`)의 모델 속을 **반대 지침(`snake_case`)**으로 바꿔치기해,
행동이 **반대 지침 쪽으로 넘어가는지**를 **어텐션만 / 내용만 / 둘다**로 나눠 **전 층 스윕**으로 잰다.
**내용만 바꿔도 넘어가고 어텐션만은 안 되면 → 지침도 코드와 같은 통로(내용/Value).** (step3의 지침 판)

**고정 설정:** 4모델 · 504이름(42묶음) · 무작위값 42 · 평균 덮어쓰기 · 문맥 준수6+위반6.

**바꾸는 것:** 규칙문의 지시어 단어 **첫 등장 하나**(후보열거의 같은 단어는 안 건드림 — 그게 옳음).

> **주의 1 (판정불가):** 지침이 그 모델 행동을 실제로 바꾸지 못하면(깨끗−base 차<1.0) 넘어감 계산이 무의미 →
> **판정불가로 분리**한다. step3(코드)와 달리 여기선 실제로 나올 수 있음(모델이 지침을 잘 안 따르면).
> **주의 2 (부하·메모리):** 전 층×3방식이라 무겁다. 84조건/모델. **deepseek-6.7b는 T4 OOM 가능** → 셀 ③ 8bit.

**모델 하나씩.** 셀 ③ `PICK` → 셀 ④~⑦. 끝나면 `PICK` 바꿔 반복. 끊겨도 저장된 건 건너뜀.


In [ ]:
# ① 환경 설정 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q transformers accelerate torch matplotlib pandas bitsandbytes

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# ② 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
BRANCH = 'integration/step1-5'
!git fetch --quiet origin $BRANCH
!git checkout $BRANCH
!git pull --quiet origin $BRANCH
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# ③ 조건 설정 — 4모델 중 하나 골라 42묶음 x 지침 방향 2종
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation, Intervention, InterventionKind)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
# deepseek-6.7b OOM나면 아래 해제(8bit):
# if MODEL.family=='deepseek': MODEL = ModelSpec(name=MODEL.name, family='deepseek', dtype='float16', quantization='8bit')

DIRECTIONS = [Notation.CAMEL, Notation.SNAKE]   # 지침이 camel / snake
BLOCKS = list(range(42))

def cond(target, block):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=6, n_functions=12, composition=Composition.POOL, pool_block=block),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        intervention=Intervention(kind=InterventionKind.KEY_VALUE, layers='sweep', target='instruction', donor='opposite'),
        token_unit='mean', seed=42)

conditions = [cond(t, b) for t in DIRECTIONS for b in BLOCKS]
print('모델:', MODEL.family, '| 조건 수:', len(conditions), '(=방향2 x 묶음42)')
print('예:', conditions[0].slug())


In [ ]:
# ④ 실행 — 개입(전 층 스윕). 조건마다 즉시 저장(재개). 지침 지시어 KV 치환.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model
import numpy as np

STEP = 'step5_instr-cause'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers} | GQA {handle.gqa_info()}')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle)   # 개입(전 층 스윕) — 넘어감(전이) 측정
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ3'))
        if i % 5 == 0 or i == len(todo):
            pl = out.metrics.per_layer; ex = out.metrics.extra
            gap = ex['S_clean'] - ex['S_base']
            cand = [(int(L), v.get('value__recovery')) for L, v in pl.items() if v.get('value__recovery') is not None]
            if abs(gap) < 1.0:
                print(f'    [{i}/{len(todo)}] 지침 레버 약함(판정불가 후보) 깨끗-base 차 {gap:+.2f}')
            elif cand:
                bL, bV = max(cand, key=lambda t: t[1])
                print(f'    [{i}/{len(todo)}] 내용만 최고 넘어감 {bV:.2f} @L{bL}  (깨끗-base 차 {gap:+.2f})')
    print('  완료.')
else:
    print('  이미 다 됨')


In [ ]:
# ⑤ 결과 로드 (이 모델 것)
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step5_instr-cause')) for c in conditions
        if result_path(c, step='step5_instr-cause').exists()]
print('불러온 조건:', len(recs), '-> results/step5_instr-cause/')


In [ ]:
# ⑥ 요약 — '어텐션만/내용만/둘다' 넘어감을 층별 평균 (판정불가 분리) + 그림
import numpy as np, pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

WAY_EN = {'key':'Attention only', 'value':'Content only', 'key_value':'Both'}
GAP_MIN = 1.0   # 깨끗-base 차가 이보다 작으면 '판정불가'(넘어감 무의미 = 지침 레버 없음)

agg = defaultdict(lambda: defaultdict(list))
n_valid = 0; n_und = 0
for r in recs:
    ex = r.metrics.extra; gap = ex['S_clean'] - ex['S_base']
    if abs(gap) < GAP_MIN:
        n_und += 1; continue
    n_valid += 1
    for L, v in r.metrics.per_layer.items():
        L = int(L)
        for way in ['key','value','key_value']:
            rr = v.get(f'{way}__recovery')
            if rr is not None: agg[way][L].append(rr)

print(f'유효 묶음 {n_valid} / 판정불가 {n_und} (지침 레버 약함)')
if n_valid == 0:
    print('!! 이 모델은 지침 레버가 약해 전부 판정불가 — 넘어감 해석 불가')
else:
    def curve(way):
        d = agg[way]; L = sorted(d)
        m = np.array([np.mean(d[l]) for l in L])
        ci = np.array([1.96*np.std(d[l],ddof=1)/np.sqrt(len(d[l])) if len(d[l])>1 else 0 for l in L])
        return np.array(L), m, ci
    Lv, mv, cv = curve('value'); bi = int(mv.argmax())
    print(f'[핵심] 내용만: 최고 넘어감 {mv[bi]:.2f} @L{Lv[bi]} (전체 {len(Lv)}층)')
    Lk, mk, ck = curve('key'); print(f'       어텐션만: 최고 넘어감 {mk.max():.2f}')
    fig, ax = plt.subplots(figsize=(7.5,4.3))
    for way,color in [('value','#2563eb'),('key','#9ca3af'),('key_value','#ea580c')]:
        L,m,ci = curve(way)
        ax.plot(L,m,color=color,lw=2,label=WAY_EN[way]); ax.fill_between(L,m-ci,m+ci,color=color,alpha=.18)
    ax.axhline(0,color='k',lw=.6); ax.set_xlabel('Layer edited'); ax.set_ylabel('Transition (0=none, 1=full flip)')
    ax.set_title(f'{MODEL.family} — step5 instruction causal (transition by layer)')
    ax.grid(alpha=.25); ax.legend(fontsize=8); plt.tight_layout(); plt.show()


In [ ]:
STEPS = ['step5_instr-cause']
# ⑧ 결과 zip으로 묶어 내려받기
import shutil, os, glob

def pack(step):
    d = f'results/{step}'
    if not os.path.isdir(d):
        print(f'  [건너뜀] {d} 폴더가 없다 — 이 스텝은 아직 안 돌렸다')
        return None
    n = len(glob.glob(f'{d}/*.json'))
    if n == 0:
        print(f'  [건너뜀] {d} 가 비어 있다')
        return None
    path = shutil.make_archive(step, 'zip', d)
    print(f'  {step}: {n}개 → {path} ({os.path.getsize(path)/1e6:.1f}MB)')
    return path

print('results/ 안에 있는 폴더:', sorted(os.listdir('results')) if os.path.isdir('results') else '(results 폴더 없음)')
print()
made = [p for p in (pack(s) for s in STEPS) if p]

if not made:
    print('\n내려받을 것이 없다. 실행 셀을 먼저 돌렸는지 확인할 것.')
else:
    try:
        from google.colab import files
        for p in made:
            files.download(p)
        print('\n다운로드 시작. 브라우저가 막으면 왼쪽 **파일 탐색기**에서 직접 받으면 된다.')
    except Exception as e:
        print(f'\nColab 자동 다운로드 불가({type(e).__name__}). 위 경로에서 직접 받을 것.')
